# Задача 10. Анализ доставки и SLA

**Постановка задачи:** операционная команда хочет понять, где чаще возникают задержки доставки. Нужно посчитать по городам и службам доставки:

- количество доставок;
- среднее фактическое время доставки;
- средний обещанный срок;
- долю доставок в SLA;
- долю задержек.

Доставка считается в SLA, если фактическое число дней меньше или равно `promised_days`.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH delivery AS (
    SELECT
        c.city,
        s.delivery_provider,
        s.order_id,
        CAST(julianday(s.delivery_date) - julianday(s.shipment_date) AS INTEGER) AS actual_days,
        s.promised_days,
        CASE
            WHEN CAST(julianday(s.delivery_date) - julianday(s.shipment_date) AS INTEGER) <= s.promised_days THEN 1
            ELSE 0
        END AS is_sla_hit
    FROM shipments s
    JOIN orders o ON s.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status IN ('delivered', 'refunded')
)
SELECT
    city,
    delivery_provider,
    COUNT(order_id) AS shipments,
    ROUND(AVG(actual_days), 2) AS avg_actual_days,
    ROUND(AVG(promised_days), 2) AS avg_promised_days,
    ROUND(100.0 * AVG(is_sla_hit), 2) AS sla_hit_rate_pct,
    ROUND(100.0 * (1 - AVG(is_sla_hit)), 2) AS delay_rate_pct
FROM delivery
GROUP BY city, delivery_provider
HAVING COUNT(order_id) >= 10
ORDER BY delay_rate_pct DESC, shipments DESC;
"""

df = pd.read_sql_query(query, conn)
df

**Комментарий стажёра:** добавил `HAVING COUNT(order_id) >= 10`, чтобы убрать слишком маленькие группы, где процент задержек может быть случайным.